# 06 Compare models
Accuracy, macro F1, params, file size, CPU ms/frame. Expected story: CNN is smaller; MobileNetV2 usually generalizes better; GRU is small and background-independent.


In [ ]:
# 1) Mount Google Drive (a popup asks permission)
from google.colab import drive
drive.mount("/content/drive")

# 2) Paths
DRIVE = "/content/drive/MyDrive/sign-language"

# 3) Code + packages
!git clone https://github.com/<you>/sign-language-translator.git /content/repo 2>/dev/null || git -C /content/repo pull
!pip install -q mediapipe==0.10.14 "numpy<2" scikit-learn seaborn tqdm
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")



In [ ]:
import os, time, json
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score, accuracy_score

from src.config import ALPHABET_LABELS, CNN_IMG_SIZE, MOBILENET_IMG_SIZE, PHRASE_LABELS, SEQ_LEN, N_FEATURES

def size_mb(p):
    return os.path.getsize(p) / 1e6 if os.path.exists(p) else None

def cpu_ms(model, dummy, n=30):
    model.predict(dummy, verbose=0)
    t0 = time.perf_counter()
    for _ in range(n):
        model.predict(dummy, verbose=0)
    return (time.perf_counter() - t0) * 1000 / n

rows = []
cnn_path = f"{DRIVE}/models/cnn.keras"
mn_path = f"{DRIVE}/models/mobilenet.keras"
gru_path = f"{DRIVE}/models/phrases_gru.keras"

if os.path.exists(cnn_path):
    m = tf.keras.models.load_model(cnn_path)
    dummy = np.zeros((1, CNN_IMG_SIZE, CNN_IMG_SIZE, 3), np.float32)
    rows.append(("cnn", m.count_params(), size_mb(cnn_path), cpu_ms(m, dummy)))
if os.path.exists(mn_path):
    m = tf.keras.models.load_model(mn_path)
    dummy = np.zeros((1, MOBILENET_IMG_SIZE, MOBILENET_IMG_SIZE, 3), np.float32)
    rows.append(("mobilenet", m.count_params(), size_mb(mn_path), cpu_ms(m, dummy)))
if os.path.exists(gru_path):
    m = tf.keras.models.load_model(gru_path)
    dummy = np.zeros((1, SEQ_LEN, N_FEATURES), np.float32)
    rows.append(("phrases_gru", m.count_params(), size_mb(gru_path), cpu_ms(m, dummy)))

print("model, params, MB, CPU ms/frame")
for r in rows:
    print(r)

print("""
Limitations: single-signer bias, lighting, similar letters (M/N/S/T/A/E), J and Z motion,
small phrase vocabulary, not a full sign language translator (no grammar, facial expressions).
Future work: more phrases via WLASL, transformer on landmarks, on-device TF.js/ONNX, other sign languages (ISL).
""")

